# Chapter 3: Five Patterns, Five Trade-offs
## Reasoning Architecture for Production Systems

**Course**: INFO 7375 — Prompt Engineering for Generative AI, Northeastern University
**Author**: Vrushti Shah | **Book**: *Design of Agentic Systems with Case Studies*

---

> **No API key required.** This notebook uses realistic pre-scripted mock responses
> that faithfully simulate Claude's ReAct and Plan-and-Execute behavior.
> Both failure modes are triggered deterministically.

**What this notebook demonstrates:**
1. ReAct vs Plan-and-Execute on the same 8-step warranty-claim task
2. **Failure Case 1** — ReAct context saturation (reasoning degrades at step 6+)
3. **Failure Case 2** — Plan-and-Execute stale-plan silent failure
4. **Mandatory Human Decision Node** — documented AI architectural rejection
5. **Defense Architecture** — re-planning trigger that recovers from failure


## Setup — No API Key Needed

In [ ]:
# No API key required — this notebook uses mock responses
# that simulate realistic Claude behavior for both patterns.
import json
from collections import Counter

print("✅ Setup complete — no API key needed")
print("   All Claude responses are pre-scripted and deterministic.")
print("   Both failure modes trigger on every run.")


## Tool Registry

Eight mock tools simulate a real warranty-claim workflow.
`check_inventory` can be **forced to fail** — this powers Failure Case 2.


In [ ]:
FORCE_INVENTORY_FAIL = False   # Set True for Failure Case 2
CALL_LOG = []

def lookup_order(order_id):
    CALL_LOG.append("lookup_order")
    return {"order_id": order_id, "customer_id": "CUST-9021",
            "product_sku": "WH-2200", "product_name": "Wireless Headphones Pro",
            "purchase_date": "2024-01-15", "warranty_months": 12, "status": "delivered"}

def check_extended_warranty(order_id):
    CALL_LOG.append("check_extended_warranty")
    return {"extended": True, "plan": "Premium 2-Year",
            "expiry": "2026-01-15", "covers_replacement": True}

def check_inventory(product_sku):
    CALL_LOG.append("check_inventory")
    if FORCE_INVENTORY_FAIL:
        raise Exception(f"InventoryServiceUnavailable: Cannot connect for SKU {product_sku}")
    return {"product_sku": product_sku, "in_stock": True,
            "quantity": 14, "warehouse": "BOS-01", "estimated_ship_days": 2}

def calculate_reimbursement(order_id, policy="standard"):
    CALL_LOG.append("calculate_reimbursement")
    return {"order_id": order_id, "original_price": 149.99,
            "reimbursement_amount": 149.99, "method": "original_payment_method"}

def verify_shipping_address(customer_id):
    CALL_LOG.append("verify_shipping_address")
    return {"customer_id": customer_id,
            "address": "42 Huntington Ave, Boston MA 02116", "verified": True}

def check_fraud_flag(order_id):
    CALL_LOG.append("check_fraud_flag")
    return {"order_id": order_id, "fraud_flag": False,
            "risk_score": 0.04, "cleared": True}

def draft_resolution_email(order_id, resolution):
    CALL_LOG.append("draft_resolution_email")
    return {"email_id": f"EMAIL-{order_id}-001", "status": "drafted",
            "preview": f"Dear Customer, your claim for {order_id} is approved. {resolution[:60]}"}

def log_crm_case(order_id, resolution):
    CALL_LOG.append("log_crm_case")
    return {"case_id": f"CASE-{order_id}", "status": "logged",
            "summary": resolution[:100]}

TOOL_FUNCTIONS = {
    "lookup_order": lookup_order,
    "check_extended_warranty": check_extended_warranty,
    "check_inventory": check_inventory,
    "calculate_reimbursement": calculate_reimbursement,
    "verify_shipping_address": verify_shipping_address,
    "check_fraud_flag": check_fraud_flag,
    "draft_resolution_email": draft_resolution_email,
    "log_crm_case": log_crm_case,
}

def call_tool(name, inputs):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"Unknown tool: {name}"}
    try:
        return TOOL_FUNCTIONS[name](**{k: v for k, v in inputs.items()
                                       if k in TOOL_FUNCTIONS[name].__code__.co_varnames})
    except Exception as e:
        return {"error": str(e), "tool": name}

print(f"✅ {len(TOOL_FUNCTIONS)} tools registered:", list(TOOL_FUNCTIONS.keys()))


## Mock Claude Responses

These pre-scripted responses faithfully simulate how Claude behaves inside
a ReAct loop. Notice how the reasoning in steps 6-8 becomes less grounded
in earlier observations — that is the context saturation effect modelled here.


In [ ]:
# ── REACT MOCK RESPONSES (simulates Claude's Thought-Action-Observation loop) ──
# Each entry is one ReAct step: the model's thought + which tool to call
REACT_STEPS_3 = [
    {"thought": "I need to look up order #4821 to check its warranty status.",
     "tool": "lookup_order", "inputs": {"order_id": "4821"}},
    {"thought": "Order found. Purchase date is 2024-01-15, warranty is 12 months — that's expired. Let me check if there's an extended warranty plan.",
     "tool": "check_extended_warranty", "inputs": {"order_id": "4821"}},
    {"thought": "Extended warranty is active until 2026-01-15. The customer is covered. I can now summarize eligibility.",
     "tool": None,
     "final": "The customer's standard 12-month warranty has expired, but their Premium 2-Year extended plan is active until January 2026. They are fully eligible for warranty service including replacement."},
]

# 8-step version — notice steps 6-8 show signs of context saturation:
# the model stops referencing earlier specific observations
REACT_STEPS_8 = [
    {"thought": "I need to process a complete warranty claim for order #4821. Starting with order lookup.",
     "tool": "lookup_order", "inputs": {"order_id": "4821"}},

    {"thought": "Order confirmed: WH-2200 purchased 2024-01-15, 12-month warranty. Checking extended warranty.",
     "tool": "check_extended_warranty", "inputs": {"order_id": "4821"}},

    {"thought": "Extended warranty active until 2026-01-15. Now checking inventory for replacement unit WH-2200.",
     "tool": "check_inventory", "inputs": {"product_sku": "WH-2200"}},

    {"thought": "Inventory available (14 units, BOS-01). Calculating reimbursement amount.",
     "tool": "calculate_reimbursement", "inputs": {"order_id": "4821", "policy": "standard"}},

    {"thought": "Reimbursement: $149.99 to original payment method. Verifying shipping address.",
     "tool": "verify_shipping_address", "inputs": {"customer_id": "CUST-9021"}},

    # ── CONTEXT SATURATION BEGINS HERE ────────────────────────────────────
    # Step 6: Model's reasoning becomes less specific — stops citing exact values
    # from earlier observations. This is the softmax dilution effect.
    {"thought": "Address verified. Checking fraud flags for the order.",
     "tool": "check_fraud_flag", "inputs": {"order_id": "4821"}},

    # Step 7: Notice — model drafts email without referencing the specific
    # inventory location (BOS-01) or exact ship date it received in step 3.
    # Early observations have been attenuated.
    {"thought": "No fraud flags. Drafting resolution email. The customer is eligible for a replacement.",
     "tool": "draft_resolution_email",
     "inputs": {"order_id": "4821",
                "resolution": "Your warranty claim is approved. A replacement unit will be shipped."}},

    # Step 8: Model re-calls lookup_order — REPEATED CALL.
    # This is the architectural signature of context saturation:
    # the model has lost track of what it already retrieved.
    {"thought": "Need to confirm order details before logging the case.",
     "tool": "lookup_order",   # ← REPEATED CALL (already called in step 1)
     "inputs": {"order_id": "4821"},
     "saturation_flag": True},

    {"thought": "Logging case in CRM.",
     "tool": "log_crm_case",
     "inputs": {"order_id": "4821",
                "resolution": "Warranty claim approved. Replacement shipped."}},

    {"thought": None,
     "tool": None,
     "final": "Warranty claim for order #4821 has been processed. Extended warranty confirmed active. Replacement unit approved for shipment. Case logged in CRM. NOTE: Step 6-8 reasoning was less grounded in specific earlier observations — characteristic of context saturation."},
]

# ── PLANNER MOCK RESPONSE ──────────────────────────────────────────────────
# This is what the AI planner generates — note: inventory BEFORE fraud check
# (data-dependency ordering, not business-rule ordering)
MOCK_PLAN = {
    "plan_summary": "8-step warranty claim processing for order #4821",
    "steps": [
        {"step_number": 1, "tool": "lookup_order",
         "inputs": {"order_id": "4821"}, "output_key": "step_1_result"},
        {"step_number": 2, "tool": "check_extended_warranty",
         "inputs": {"order_id": "4821"}, "output_key": "step_2_result"},
        {"step_number": 3, "tool": "check_inventory",       # ← AI put this BEFORE fraud check
         "inputs": {"product_sku": "WH-2200"}, "output_key": "step_3_result"},
        {"step_number": 4, "tool": "calculate_reimbursement",
         "inputs": {"order_id": "4821"}, "output_key": "step_4_result"},
        {"step_number": 5, "tool": "verify_shipping_address",
         "inputs": {"customer_id": "CUST-9021"}, "output_key": "step_5_result"},
        {"step_number": 6, "tool": "check_fraud_flag",      # ← fraud AFTER resource allocation
         "inputs": {"order_id": "4821"}, "output_key": "step_6_result"},
        {"step_number": 7, "tool": "draft_resolution_email",
         "inputs": {"order_id": "4821",
                    "resolution": "Claim approved. Replacement shipping within 2 days."},
         "output_key": "step_7_result"},
        {"step_number": 8, "tool": "log_crm_case",
         "inputs": {"order_id": "4821",
                    "resolution": "Warranty claim approved, replacement dispatched."},
         "output_key": "step_8_result"},
    ]
}

# ── RECOVERY PLAN (used by defense architecture cell) ─────────────────────
MOCK_RECOVERY_PLAN = {
    "plan_summary": "Recovery plan — skip failed inventory step, proceed with reimbursement only",
    "steps": [
        {"step_number": 4, "tool": "calculate_reimbursement",
         "inputs": {"order_id": "4821", "policy": "standard"}, "output_key": "step_4_result"},
        {"step_number": 5, "tool": "verify_shipping_address",
         "inputs": {"customer_id": "CUST-9021"}, "output_key": "step_5_result"},
        {"step_number": 6, "tool": "check_fraud_flag",
         "inputs": {"order_id": "4821"}, "output_key": "step_6_result"},
        {"step_number": 7, "tool": "draft_resolution_email",
         "inputs": {"order_id": "4821",
                    "resolution": "Claim approved for reimbursement. Replacement pending inventory restore."},
         "output_key": "step_7_result"},
        {"step_number": 8, "tool": "log_crm_case",
         "inputs": {"order_id": "4821",
                    "resolution": "Warranty approved. Reimbursement issued. Replacement on backorder."},
         "output_key": "step_8_result"},
    ]
}

print("✅ Mock responses loaded")
print(f"   ReAct 3-step : {len(REACT_STEPS_3)} steps")
print(f"   ReAct 8-step : {len(REACT_STEPS_8)} steps (includes saturation at step 6+)")
print(f"   Plan steps   : {len(MOCK_PLAN['steps'])} steps")
print(f"   Recovery plan: {len(MOCK_RECOVERY_PLAN['steps'])} steps")


## ReAct Agent (Mock)

Runs the pre-scripted ReAct steps, executes real tool calls,
and accumulates a growing `messages` list — exactly as the real Claude API would.
The context saturation effect is visible in the step 6-8 reasoning.


In [ ]:
def react_agent(task, steps_script, verbose=True):
    """
    ReAct agent using mock Claude responses.
    Executes real tool calls against the mock tool registry.
    Accumulates context exactly as the live version would.
    """
    CALL_LOG.clear()
    messages = [{"role": "user", "content": task}]
    step_num = 0

    if verbose:
        print(f"\n{'='*60}")
        print(f"ReAct Agent — {len([s for s in steps_script if s.get('tool')])} tool calls")
        print(f"{'='*60}")

    for step in steps_script:
        step_num += 1
        ctx_tokens = sum(len(str(m)) for m in messages) // 4

        if verbose:
            print(f"\n[Step {step_num}]  Context: ~{ctx_tokens:,} tokens")

        thought = step.get("thought")
        if thought and verbose:
            saturation = " ⚠️  SATURATION FLAG" if step.get("saturation_flag") else ""
            print(f"  THOUGHT: {thought}{saturation}")

        # Final answer — no more tool calls
        if step.get("final"):
            messages.append({"role": "assistant",
                             "content": [{"type": "text", "text": step["final"]}]})
            if verbose:
                print(f"\n✅ Completed in {step_num} steps")
                print(f"   Answer: {step['final'][:200]}")
            break

        # Execute tool call
        tool_name = step.get("tool")
        if tool_name:
            inputs = step.get("inputs", {})
            if verbose:
                repeated = CALL_LOG.count(tool_name) > 0
                repeat_badge = " 🔴 REPEATED CALL — context saturation signature" if repeated else ""
                print(f"  ACTION: {tool_name}({json.dumps(inputs)[:50]}){repeat_badge}")

            result = call_tool(tool_name, inputs)

            if verbose:
                print(f"  OBS:    {str(result)[:80]}")

            # Append to messages — context grows with every step
            messages.append({"role": "assistant", "content": [
                {"type": "text", "text": thought or ""},
                {"type": "tool_use", "id": f"tool_{step_num}",
                 "name": tool_name, "input": inputs}
            ]})
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": f"tool_{step_num}",
                 "content": json.dumps(result)}
            ]})

    return {
        "steps": step_num,
        "call_log": list(CALL_LOG),
        "messages": messages,
        "completed": True,
    }

print("✅ ReAct agent defined")


## Plan-and-Execute Agent (Mock)

Uses the mock plan directly (no API call needed).
The executor runs real tool calls and handles failures exactly as described in the chapter.


In [ ]:
def execute_plan(plan, verbose=True):
    """
    Execute a plan step by step.
    When a step fails: error goes into context, execution continues.
    No re-planning. No exception raised. This is Failure Case 2.
    """
    CALL_LOG.clear()
    context = {}
    log = []
    had_error = False

    if verbose:
        print(f"\n[EXECUTING — {len(plan['steps'])} steps]")

    for step in plan["steps"]:
        resolved = {}
        for k, v in step["inputs"].items():
            resolved[k] = context.get(v, v)

        if verbose:
            print(f"\n  Step {step['step_number']}: {step['tool']}"
                  f"({json.dumps(resolved)[:55]})")

        result = call_tool(step["tool"], resolved)

        if "error" in result:
            had_error = True
            context[step["output_key"]] = result
            log.append({"step": step["step_number"], "tool": step["tool"],
                        "status": "ERROR", "result": result})
            if verbose:
                print(f"  ⚠️  ERROR: {result['error']}")
                print(f"      → Error value stored in context. Execution continues.")
        else:
            status = "CORRUPTED" if had_error else "OK"
            context[step["output_key"]] = result
            log.append({"step": step["step_number"], "tool": step["tool"],
                        "status": status, "result": result})
            if verbose:
                if had_error:
                    print(f"  ⚠️  RAN ON CORRUPTED INPUT: {str(result)[:60]}")
                else:
                    print(f"  ✓  {str(result)[:70]}")

    return {"context": context, "execution_log": log, "call_log": list(CALL_LOG)}


def plan_and_execute_agent(plan=None, verbose=True):
    if plan is None:
        plan = MOCK_PLAN
    if verbose:
        print(f"\n[PLAN]  {plan['plan_summary']}")
        for s in plan["steps"]:
            print(f"  Step {s['step_number']}: {s['tool']}()  →  {s['output_key']}")
    return execute_plan(plan, verbose=verbose)

print("✅ Plan-and-Execute agent defined")


---
## 🔴 Mandatory Human Decision Node

> **Read and complete this cell before running any agent.**

### What the AI Planner Proposed
The AI generated this step order based on **data dependencies**:
`lookup → warranty → inventory (step 3) → calculate → address → fraud (step 6) → email → log`

**AI reasoning**: *"inventory check must follow lookup because the product SKU is a required input."*

### What I Rejected — and Why
**My rejection**: This ordering performs **resource allocation** (inventory reservation at step 3)
**before fraud clearance** (step 6). If fraud check fails at step 6, inventory has already
been reserved for a potentially fraudulent claim.

**Correct production order**:
`lookup → warranty → fraud → address → inventory → calculate → email → log`

**Architectural lesson**: The model optimized for data flow. I must optimize for business risk.
This is a decision the model cannot make from training data alone.


In [ ]:
# ── MANDATORY HUMAN DECISION NODE ─────────────────────────────────────────
# The AI planner proposed inventory before fraud check (data-dependency order).
# A human must verify business-rule ordering before execution proceeds.
#
# VERIFY BEFORE PROCEEDING:
#   [ ] Business-rule ordering reviewed (not just data-dependency ordering)
#   [ ] Tool failure rates assessed
#   [ ] Re-planning trigger decision documented

human_decision = {
    "plan_approved": True,
    "reordering_required": True,
    "replanning_trigger_added": False,  # Intentionally OFF to show Failure Case 2
    "architect_note": (
        "AI proposed inventory at step 3 (data-dependency: needs product SKU). "
        "REJECTED: business rule requires fraud clearance before resource allocation. "
        "Re-planning trigger disabled in this demo to demonstrate the failure mode."
    )
}

print("=" * 55)
print("  AI SCAFFOLD HALTED — HUMAN DECISION REQUIRED")
print("=" * 55)
print("  The planner proposed an execution order.")
print("  A human must verify business-rule ordering")
print("  before execution is allowed to proceed.")
print()
print("  AI proposed  : inventory at step 3 (data dependency)")
print("  Human review : REJECTED — fraud must precede inventory")
print("  Decision     : APPROVED with reordering noted")
print("=" * 55)
print()
print("Documented decision:")
print(json.dumps(human_decision, indent=2))


---
## Demo 1 — Happy Path (3-step task)
Both patterns succeed on a simplified 3-step task.
**This is the test environment that passed QA.**


In [ ]:
SIMPLE_TASK = (
    "Process a warranty claim for order #4821. "
    "Look up the order, check extended warranty, and summarize eligibility."
)

FORCE_INVENTORY_FAIL = False

print("="*60)
print("HAPPY PATH  —  3-step task (both patterns succeed)")
print("="*60)

print("\n--- ReAct (3-step) ---")
react_simple = react_agent(SIMPLE_TASK, REACT_STEPS_3, verbose=True)

print("\n\n--- Plan-and-Execute (3-step) ---")
pe_simple = plan_and_execute_agent(
    plan={"plan_summary": "3-step lookup",
          "steps": MOCK_PLAN["steps"][:3]}, verbose=True)

print("\n" + "="*60)
print("Both patterns completed the 3-step task.")
print("Production runs 8 steps. That is where the difference appears.")
print("="*60)


---
## 🔴 Failure Case 1 — ReAct Context Saturation

**Task**: Full 8-step warranty claim
**What to watch**: Repeated tool calls (step 8 re-calls lookup_order already called at step 1),
reasoning that becomes less grounded in specific earlier observations at steps 6–8
**Why it happens**: Context accumulates 3+ blocks per step. By step 6, softmax attention
dilutes early observations. The model pattern-matches the ReAct format rather than
reasoning from evidence.


In [ ]:
FULL_TASK = (
    "Process a COMPLETE warranty claim for order #4821. "
    "Complete all 8 steps: lookup, warranty check, inventory, reimbursement, "
    "address verification, fraud check, draft email, log CRM case."
)

FORCE_INVENTORY_FAIL = False
CALL_LOG.clear()

print("="*60)
print("FAILURE CASE 1  —  ReAct on 8-step task")
print("Watch: repeated calls and context saturation at step 6+")
print("="*60)

react_full = react_agent(FULL_TASK, REACT_STEPS_8, verbose=True)

print("\n" + "="*60)
print("ANALYSIS")
print("="*60)
call_counts = Counter(react_full["call_log"])
repeated    = {k: v for k, v in call_counts.items() if v > 1}
ctx_tokens  = sum(len(str(m)) for m in react_full["messages"]) // 4

print(f"  Steps taken   : {react_full['steps']}")
print(f"  Context (end) : ~{ctx_tokens:,} tokens")
print(f"  All calls     : {react_full['call_log']}")
print(f"  Unique tools  : {len(set(react_full['call_log']))}/8")

if repeated:
    print(f"\n⚠️  REPEATED CALLS DETECTED: {repeated}")
    print("   Architectural cause: softmax attention dilution on early observations.")
    print("   Model re-invoked lookup_order at step 8 — already called at step 1.")
    print("   Early observations attenuated in 8,000+ token context.")
    print("   This is context saturation. Architecture caused this, not the model.")
else:
    print("\n✓  No repeated calls detected.")


---
## 🔴 Failure Case 2 — Plan-and-Execute Stale Plan

**Injected failure**: `check_inventory` throws at step 3
**What to watch**: Error at step 3, steps 4–8 marked CORRUPTED, zero exceptions raised
**Why it happens**: Executor committed to the plan. No re-planning trigger.
The Happy Path Assumption failed silently.


In [ ]:
FORCE_INVENTORY_FAIL = True   # Inject the failure
CALL_LOG.clear()

print("="*60)
print("FAILURE CASE 2  —  Plan-and-Execute with tool failure at step 3")
print("FORCE_INVENTORY_FAIL = True")
print("="*60)

pe_full = plan_and_execute_agent(verbose=True)

print("\n" + "="*60)
print("ANALYSIS")
print("="*60)
errors    = sum(1 for e in pe_full["execution_log"] if e["status"] == "ERROR")
corrupted = sum(1 for e in pe_full["execution_log"] if e["status"] == "CORRUPTED")
total     = len(pe_full["execution_log"])

print(f"  Total steps executed      : {total}")
print(f"  Steps that errored        : {errors}")
print(f"  Steps on corrupted input  : {corrupted}")
print(f"  Exceptions raised         : 0  ← no crash; wrong output silently produced")
print()
print("ARCHITECTURAL DIAGNOSIS:")
print("  Executor committed to plan before any tools were called.")
print("  Step 3 failure → error dict placed in context.")
print("  Steps 4–8 ran on that error dict as if it were valid inventory data.")
print("  Resolution email drafted. CRM case logged. Output is wrong.")
print("  This is the Happy Path Assumption failing silently.")

FORCE_INVENTORY_FAIL = False  # Reset


---
## Defense Architecture — Re-planning Trigger

Same injected failure as Failure Case 2.
One change: when a step errors, the planner is called again with remaining steps.
The system recovers.

> **Same model. Same failure. Different architecture. Different outcome.**


In [ ]:
def execute_plan_with_defense(plan, verbose=True):
    """
    Plan-and-Execute WITH a re-planning trigger.
    When a step fails, generate a recovery plan for remaining steps.
    """
    CALL_LOG.clear()
    context  = {}
    log      = []
    steps    = list(plan["steps"])
    replans  = 0
    i        = 0

    if verbose:
        print(f"\n[EXECUTING WITH DEFENSE — {len(steps)} steps]")

    while i < len(steps):
        step = steps[i]
        resolved = {k: context.get(v, v) for k, v in step["inputs"].items()}

        if verbose:
            print(f"\n  Step {step['step_number']}: {step['tool']}"
                  f"({json.dumps(resolved)[:50]})")

        result = call_tool(step["tool"], resolved)

        if "error" in result:
            replans += 1
            if verbose:
                print(f"  ⚠️  ERROR: {result['error']}")
                print(f"  → RE-PLANNING triggered (replan #{replans})")

            # Use mock recovery plan (simulates what a real replanner would return)
            remaining = MOCK_RECOVERY_PLAN
            steps = steps[:i] + remaining["steps"]
            context[step["output_key"]] = result
            log.append({"step": step["step_number"], "tool": step["tool"],
                        "status": "ERROR+REPLANNED", "result": result})
            if verbose:
                print(f"  ✓  Recovery plan loaded: {len(remaining['steps'])} steps remaining")
        else:
            context[step["output_key"]] = result
            log.append({"step": step["step_number"], "tool": step["tool"],
                        "status": "OK", "result": result})
            if verbose:
                print(f"  ✓  {str(result)[:70]}")
        i += 1

    return {"context": context, "execution_log": log,
            "call_log": list(CALL_LOG), "replans": replans}


# ── RUN ────────────────────────────────────────────────────────────────────
FORCE_INVENTORY_FAIL = True   # Same failure as Case 2
CALL_LOG.clear()

print("="*60)
print("DEFENSE ARCHITECTURE  —  Re-planning trigger active")
print("FORCE_INVENTORY_FAIL = True  (same failure as Failure Case 2)")
print("="*60)

defense_result = execute_plan_with_defense(MOCK_PLAN, verbose=True)

print("\n" + "="*60)
print("ANALYSIS — Defense vs No Defense")
print("="*60)
ok_d   = sum(1 for e in defense_result["execution_log"] if e["status"] == "OK")
err_d  = sum(1 for e in defense_result["execution_log"] if "ERROR" in e["status"])
print(f"  Re-planning triggers fired : {defense_result['replans']}")
print(f"  Steps completed OK         : {ok_d}")
print(f"  Steps that errored         : {err_d}")
print()
print("WITHOUT defense (Failure Case 2) : silent wrong output, 0 exceptions")
print("WITH    defense (this cell)      : re-planned, recovered, correct output")
print()
print("LESSON: One architectural decision — adding a re-planning trigger —")
print("        changed the outcome. Same model. Same failure. Same tools.")
print("        Architecture is the leverage point.")

FORCE_INVENTORY_FAIL = False


---
## Summary — Architecture Determines the Failure

In [ ]:
print("="*65)
print(f"{'ARCHITECTURAL COMPARISON':^65}")
print("="*65)
print(f"  {'Dimension':<30} {'ReAct':<17} {'Plan-and-Execute'}")
print("-"*65)
rows = [
    ("Failure mode",          "Context saturation",  "Stale plan exec"),
    ("Fails at step ~",       "6-8 (8-step task)",   "First tool fail"),
    ("Error visibility",      "Gradual degrade",     "Silent propagate"),
    ("Adaptation",            "High (each step)",    "None (plan only)"),
    ("Parallelizable",        "No",                  "Yes"),
    ("Human checkpoint",      "Awkward to add",      "Natural post-plan"),
    ("Best task depth",       "1-5 steps",           "5+ steps"),
    ("Exceptions on failure", "Possible",            "0 — silent"),
]
for r in rows:
    print(f"  {r[0]:<30} {r[1]:<17} {r[2]}")
print("="*65)
print()
print("Same model. Same tools. Same task.")
print("Different architecture → different failure mode.")
print("No model upgrade fixes an architectural mismatch.")


---
## Chapter Exercises

**Exercise 1 — Understand the saturation threshold**
Look at `REACT_STEPS_8` in Cell 4. The `saturation_flag: True` marks step 8 where
the model repeats a tool call. Remove the last 3 entries (steps 6, 7, 8 with the repeat)
and re-run `react_agent` with the shortened script.
Does the repeated call disappear? At what step count does saturation appear?

Complete this sentence:
*"The failure appears at ~N steps because at that context depth, softmax attention
distributes weight across so many tokens that early observations receive less than
[X]% of total attention weight, causing the model to pattern-match the ReAct format
rather than retrieve specific evidence."*

**Exercise 2 — Move the failure point**
Change `FORCE_INVENTORY_FAIL = True` to trigger `check_fraud_flag` instead
(add a `FORCE_FRAUD_FAIL` flag to that function).
- When failure is at step 3: how many steps run on corrupted input?
- When failure is at step 6: how many steps run on corrupted input?
- What does this tell you about where to place high-reliability steps in a plan?

**Exercise 3 — Evaluate the defense**
Look at `MOCK_RECOVERY_PLAN` in Cell 4. The recovery plan skips inventory and
issues a reimbursement instead. Is this always the right recovery strategy?
What information would a real re-planner need that this mock doesn't have?
